# ColabUMA Workflow — 最適化 → スキャン → 遷移状態を1つのセッションで

[ColabUMA_Opt](https://colab.research.google.com/github/RChibaChiba/ColabChem/blob/main/ColabUMA_Opt.ipynb)（構造最適化）、[ColabUMA_Scan](https://colab.research.google.com/github/RChibaChiba/ColabChem/blob/main/ColabUMA_Scan.ipynb)（緩和スキャン）、[ColabUMA_TS](https://colab.research.google.com/github/RChibaChiba/ColabChem/blob/main/ColabUMA_TS.ipynb)（遷移状態探索）を、**同じセッションのまま続けて** 実行するノートブックです。Colab ではノートごとにセッションが分かれるので、別々のノートで続けるとそのたびにインストールとモデル読み込みをやり直すことになります。このノートなら1回で済み、前の段階の構造をそのまま次の段階に渡します。

計算の中身は3つのノートと同じ共通コード（`colabuma` パッケージ）を使っています。

## 流れ

1. **構造最適化**：入力構造を最適化する（反応物）
2. **スキャン**：最適化した構造から、指定した結合長・角度・二面角を動かす。スキャンの後に、最後の点からさかのぼった極小を最適化する（生成物）
3. **遷移状態**：反応物と生成物から DMF で TS の初期構造を作り、Sella で最適化する。「Sella from the scan maximum」を選ぶと、DMF を飛ばしてスキャンの最高点から始める
4. **DFT 一点計算**：反応物・TS・生成物のエネルギーを DFT で計算する

## 使い方

1. メニューの **ランタイム → ランタイムのタイプを変更** で **T4 GPU** を選ぶ
2. 初回のみ、左サイドバーの🔑（シークレット）に `HF_TOKEN` を登録し、「ノートブックからのアクセス」をオンにする
3. **1. Input form** を埋めて **ランタイム → すべて実行** を押し、**2.** で入力構造（xyz または gjf）を選ぶ。あとは放置できます
4. 最後のセルで、すべての段階の結果を1つの ZIP にまとめてダウンロードします

途中の段階だけやり直したいときは、フォームを直してから、そのセル以降を選んで実行してください（**ランタイム → 以降のセルを実行**）。インストールとモデル読み込みはスキップされます。

原子番号が分からないときは、一度実行すると「5. Prepare the structures」に番号付きの3D表示が出ます。

> **事前準備**：モデルの重み [facebook/UMA](https://huggingface.co/facebook/UMA) はアクセス申請が必要です。Hugging Faceにログインしてページ上でFAIR Chemistry License v1に同意し、承認後に [Access Tokens](https://huggingface.co/settings/tokens) で Read 権限のトークンを作成してください。

## 注意事項

- スキャンで結合を作る・切る方向に十分動かし、スキャン後の最適化が生成物に落ちるようにしてください。スキャン後の最適化構造が入力と同じなら、DMF は始められません。
- 各段階の注意事項は、それぞれのノートの説明を見てください。

## ライセンス

- このノートと共通コード（`colabuma`）：[MIT License](https://github.com/RChibaChiba/ColabChem/blob/main/LICENSE)
- UMAのモデル重み：[FAIR Chemistry License v1](https://huggingface.co/facebook/UMA) に従います
- 実行時にインストールするライブラリは、それぞれのライセンスに従います。一覧は [README のライセンス](https://github.com/RChibaChiba/ColabChem/blob/main/README.ja.md#ライセンス) にあります

## 引用

論文で使う場合は、以下を引用してください。

- B. M. Wood *et al.*, "UMA: A Family of Universal Models for Atoms", arXiv:2506.23971 (2025).
- D. S. Levine *et al.*, "The Open Molecules 2025 (OMol25) Dataset, Evaluations, and Models", arXiv:2505.08762 (2025).
- fairchem: https://github.com/facebookresearch/fairchem

遷移状態探索（DMF・Sella）を使った場合は、次も引用してください。

- S.-i. Koda and S. Saito, *J. Open Source Softw.* **11**, 10379 (2026).（PyDMF）
- S.-i. Koda and S. Saito, *J. Chem. Theory Comput.* **20**, 2798–2811 (2024).（Direct MaxFlux）
- S.-i. Koda and S. Saito, *J. Chem. Theory Comput.* **20**, 7176–7187 (2024); **21**, 3513–3522 (2025).（FB-ENM、CFB-ENM）
- E. D. Hermes, K. Sargsyan, H. N. Najm, J. Zádor, *J. Chem. Theory Comput.* **18**, 6974–6988 (2022).（Sella）
- DFT の一点計算を使った場合：PySCF（Q. Sun *et al.*, *J. Chem. Phys.* **153**, 024109 (2020)）と GPU4PySCF（X. Wu *et al.*, arXiv:2404.09452 (2024)）

In [ ]:
#@title 1. Input form { display-mode: "form" }
#@markdown When the form is filled in, choose **Runtime → Run all**. Pick an xyz or gjf file in the next cell; after that you can leave it running.

#@markdown ### Output
output_name = "result"  #@param {type:"string"}

#@markdown ### Structure
charge = 0  #@param {type:"integer"}
multiplicity = 1  #@param {type:"integer"}

#@markdown ### 1. Optimization
run_vibrations = False  #@param {type:"boolean"}

#@markdown ### 2. Scan
#@markdown Atom numbers (from 1). 2 atoms give a bond length, 3 an angle, 4 a dihedral. An empty start value starts from the optimized structure.
scan_atoms = "1 2"  #@param {type:"string"}
scan_start = ""  #@param {type:"string"}
scan_n_steps = 20  #@param {type:"integer"}
scan_step_size = -0.1  #@param {type:"number"}

#@markdown ### 3. Transition state
ts_source = "DMF → Sella (from the optimized and post-scan structures)"  #@param ["DMF → Sella (from the optimized and post-scan structures)", "Sella from the scan maximum"]

#@markdown DMF settings (when DMF is chosen)
#@markdown Number of points along the path where the energy is computed. More points give a better TS guess but take longer.
dmf_nmove = 5  #@param {type:"integer"}
dmf_convergence = "middle"  #@param ["loose", "middle", "tight"]
dmf_max_iter = 300  #@param {type:"integer"}

#@markdown Build the initial Hessian by numerical differentiation of UMA (6N force calls). Off starts from an approximate Hessian.
ts_exact_hessian = True  #@param {type:"boolean"}
#@markdown Displace the TS slightly along the imaginary mode, optimize both sides, and check where they lead (simple IRC check).
check_connection = True  #@param {type:"boolean"}

#@markdown ### 4. DFT single points (gpu4pyscf)
#@markdown Computes DFT energies at the UMA structures and compares activation energies. ωB97M-V/def2-TZVPD is the level of theory of UMA's training data (OMol25). def2-TZVPD can take tens of minutes per point on a T4.
run_dft = False  #@param {type:"boolean"}
dft_functional = "B3LYP-D3(BJ)"  #@param ["ωB97M-V (same as OMol25)", "B3LYP-D3(BJ)", "PBE0-D3(BJ)", "M06-2X", "r2SCAN-D4"]
dft_basis = "def2-SVP"  #@param ["def2-SVP", "def2-TZVP", "def2-TZVPD"]
#@markdown Time limit for DFT (minutes, all structures together). Past it, DFT is skipped and the UMA results are saved. 0 means no limit (DFT still stops before the Colab runtime is likely to end).
dft_time_limit = 60  #@param {type:"integer"}
#@markdown Route line of the Gaussian TS gjf put in the ZIP.
gaussian_route = "# opt=(ts,calcfc,noeigentest) freq b3lyp/def2svp empiricaldispersion=gd3bj"  #@param {type:"string"}
#@markdown Route line of the Gaussian IRC gjf put in the ZIP (the structure is the UMA TS).
gaussian_irc_route = "# irc=(calcfc,maxpoints=30,stepsize=10) b3lyp/def2svp empiricaldispersion=gd3bj"  #@param {type:"string"}

#@markdown ### Settings
model_name = "uma-s-1p1"  #@param ["uma-s-1p1", "uma-s-1p2", "uma-m-1p1"]
task_name = "omol"  #@param ["omol"]
optimizer_name = "LBFGS"  #@param ["LBFGS", "BFGS", "FIRE"]
#@markdown Convergence threshold fmax (eV/Å). normal is close to Gaussian's default, tight to Gaussian's opt=tight.
convergence = "normal (0.02)"  #@param ["loose (0.05)", "normal (0.02)", "tight (0.001)", "custom"]
#@markdown fmax (eV/Å) used when custom is chosen.
fmax_custom = 0.02  #@param {type:"number"}
#@markdown The convergence threshold and maximum steps apply to all stages (per point in the scan).
max_steps = 500  #@param {type:"integer"}

In [ ]:
#@title 2. File selection and checks { display-mode: "form" }
#@markdown File selection and the token check happen here. You can leave the install and calculations after this running.

import os
import subprocess
import sys

COLABUMA_REF = "main"  # GitHub branch or tag of the shared code (colabuma package)


def _use_colabuma(ref):
    # Use colabuma next to the notebook if it is there, otherwise fetch it from GitHub
    if os.path.isdir("colabuma"):
        path = os.getcwd()
    else:
        path = "/content/ColabChem" if os.path.isdir("/content") else os.path.expanduser("~/ColabChem")
        if not os.path.isdir(os.path.join(path, "colabuma")):
            r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", ref,
                                "https://github.com/RChibaChiba/ColabChem.git", path], capture_output=True, text=True)
            if r.returncode != 0 or not os.path.isdir(os.path.join(path, "colabuma")):
                raise RuntimeError(f"Could not fetch the ColabChem shared code from GitHub (ref {ref}).\n{r.stderr}")
    if path not in sys.path:
        sys.path.insert(0, path)


_use_colabuma(COLABUMA_REF)

from colabuma import dft, env, gaussian, opt, output, scan, structure, ts, ui
from colabuma.ui import ColabUMAError, heading, ok

ui.install_error_handler()

fmax = opt.fmax_from_form(convergence, fmax_custom)
structure.check_form(fmax, max_steps, multiplicity)
job = opt.Job(structure.out_prefix(output_name), charge, multiplicity, model_name, task_name,
              optimizer_name, fmax, max_steps, convergence)
spec = scan.parse_spec(scan_atoms, scan_start, scan_n_steps, scan_step_size, opt_before=False, opt_after=True)
use_dmf = ts_source.startswith("DMF")
if dmf_nmove < 1 or dmf_max_iter < 1:
    raise ColabUMAError("DMF's nmove and maximum iterations must be 1 or more.", "Typical values are nmove = 5 and 300 iterations.",
                        ja="DMF の nmove と最大反復回数は1以上にしてください。", ja_hint="通常は nmove = 5、最大反復回数 = 300 程度です。")
if run_dft:
    dft.parse_method(dft_functional, dft_basis)
    if dft_time_limit < 0:
        raise ColabUMAError("The DFT time limit must be 0 or more.", "0 means no limit.",
                            ja="DFT の時間の上限は0以上にしてください。", ja_hint="0 なら上限なしです。")
env.check_gpu()
env.ensure_token()
upload_name, upload_data = structure.upload()
scan.check_atom_range(spec, structure.quick_atom_count(upload_name, upload_data),
                      "Fix the form and run all again. In the file selection, Cancel upload reuses the previous file.")
ok("Input checked. You can leave it running from here.")

In [ ]:
#@title 3. Setup (install and login) { display-mode: "form" }
DEVICE = env.setup(["ts", "dft"] if run_dft else ["ts"])

In [ ]:
#@title 4. Load the model { display-mode: "form" }
calc = env.load_calculator(model_name, task_name)

In [ ]:
#@title 5. Prepare the structures { display-mode: "form" }
atoms0, source_desc = structure.read_structure(upload_name, upload_data, charge, multiplicity)
ok(f"Initial structure ready: {atoms0.get_chemical_formula()} ({len(atoms0)} atoms).")
if run_dft:
    dft.check_memory([("", atoms0)], charge, multiplicity, dft.parse_method(dft_functional, dft_basis)[1])
sc = scan.prepare(spec, atoms0)
all_files, all_logs = {}, []

In [ ]:
#@title 6. Optimization { display-mode: "form" }
heading("1. Optimization")
print(f"Optimizing with {optimizer_name} (fmax < {fmax} eV/Å, at most {max_steps} steps)")
best = opt.optimize(atoms0, calc, job, "Structure")
vib_result = opt.vibrations(best["atoms"], calc) if run_vibrations else None
_f, _l = opt.report(best, vib_result, job)
all_files.update(_f)
all_logs.append(_l)

In [ ]:
#@title 7. Scan { display-mode: "form" }
heading("2. Scan")
scan_res = scan.run(sc, best["atoms"], calc, job, skip_preopt_result=best)
_f, _l = scan.report(scan_res, job, atoms0)
for _k in ("preopt.xyz", "preopt.log"):  # same as the optimization result (opt.xyz / opt.log), so left out
    _f.pop(_k, None)
all_files.update(_f)
all_logs.append(_l)

In [ ]:
#@title 8. Transition state { display-mode: "form" }
heading("3. Transition state")
reactant = best["atoms"]
product = ts.check_endpoints(reactant, scan_res["post"]["atoms"])
endpoints = {"reactant": {"atoms": reactant, "energy": best["energy"], "converged": best["converged"]},
             "product": {"atoms": product, "energy": scan_res["post"]["energy"], "converged": scan_res["post"]["converged"]}}
dmf_res = None
if use_dmf:
    dmf_res = ts.run_dmf(reactant, product, calc, job, nmove=dmf_nmove, tol=dmf_convergence, max_iter=dmf_max_iter)
    ts_guess = dmf_res["ts_guess"]
else:
    ts_guess = scan_res["highest"]["atoms"]
    print(f"Optimizing with Sella from the scan maximum (point {scan_res['highest']['point']}).")
ts_res = ts.run_sella(ts_guess, calc, job, exact_hessian=ts_exact_hessian)
ts_vib = ts.check_ts(ts_res, calc)
conn = None
if check_connection and ts_vib is not None and ts_vib["n_imag"] >= 1:
    conn = ts.connect(ts_res["atoms"], ts_vib, calc, job, reactant, product)
_f, ts_log, uma_rows = ts.report(job, reactant, product, dmf_res, ts_res, ts_vib, conn, endpoints)
_f["ts.gjf"] = gaussian.gjf(ts_res["atoms"], job, gaussian_route, f"{job.out_prefix} TS from ColabUMA ({model_name})",
                              chk=f"{job.out_prefix}_ts.chk")
_f["irc.gjf"] = gaussian.gjf(ts_res["atoms"], job, gaussian_irc_route, f"{job.out_prefix} IRC from ColabUMA TS ({model_name})",
                               chk=f"{job.out_prefix}_irc.chk")
_guess = ts_guess.copy()  # UMA energy of the structure given to Sella
_guess.calc = calc
_star = dict(marker="*", ms=16, markeredgecolor="black")
scan.overlay_plot(scan_res, dmf_res, [
    ("TS guess (DMF max)" if use_dmf else "TS guess (scan max)", _guess, float(_guess.get_potential_energy()),
     dict(_star, color="#ff7f0e")),
    ("Sella TS", ts_res["atoms"], ts_res["energy"], dict(_star, ms=19, color="#d62728")),
    ("Product (post-scan opt)", product, scan_res["post"]["energy"], dict(marker="D", ms=8, color="#9467bd")),
])
all_files.update(_f)
all_logs.append([f"[Transition state] initial TS: {ts_source}"] + ts_log)

In [ ]:
#@title 9. DFT single points { display-mode: "form" }
heading("4. DFT single points")
if run_dft:
    _f, _l = dft.run_optional(ts.dft_points(ts_res, endpoints, conn), charge, multiplicity, dft_functional,
                              dft_basis, uma_rows, dft_time_limit)
    all_files.update(_f)
    all_logs.append(_l)
else:
    print("DFT single points are off.")

In [ ]:
#@title 10. Save { display-mode: "form" }
all_files["log.txt"] = output.log_text("ColabUMA Workflow log", source_desc, job, all_logs)
zip_path = output.save_zip(job.out_prefix, all_files)